<a href="https://colab.research.google.com/github/mbahramii/avaa-asr/blob/main/Ganjoor_Audio_Preprocessing_%26_Sharding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Dependencies & Google Drive Mount
!pip install -q datasets torchaudio huggingface_hub pandas tqdm miniaudio soundfile

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# Cell 2: Text Normalization and Dataset Splitting
import os
import re
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/farsi_asr_project"
os.makedirs(PROJECT_DIR, exist_ok=True)

# Path to the mapped catalog
RAW_CSV_PATH = os.path.join(PROJECT_DIR, "ganjoor_dataset_mapped.csv")
PREPARED_CSV_PATH = os.path.join(PROJECT_DIR, "ganjoor_prepared_metadata.csv")

if not os.path.exists(RAW_CSV_PATH) and os.path.exists("ganjoor_dataset_mapped.csv"):
    RAW_CSV_PATH = "ganjoor_dataset_mapped.csv"

print("--- 1. Loading Base Metadata ---")
df = pd.read_csv(RAW_CSV_PATH)

print("--- 2. Performing Text Normalization ---")
def normalize_persian_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('ي', 'ی').replace('ك', 'ک')
    diacritics = re.compile(r'[\u064B-\u065F\u0670]')
    text = re.sub(diacritics, '', text)
    text = re.sub(r'\u200c+', '\u200c', text)
    text = re.sub(r'[\u200e\u200f]', '', text)
    text = re.sub(r'[^\w\s\u200c]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

text_col = 'text' if 'text' in df.columns else 'transcript'
df['text_clean'] = df[text_col].apply(normalize_persian_text)
df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)

print("--- 3. Applying Shard-Level Data Split ---")
unique_tars = df['tar_file'].unique()
np.random.seed(42)
np.random.shuffle(unique_tars)

num_tars = len(unique_tars)
train_cutoff = int(num_tars * 0.90)
val_cutoff = int(num_tars * 0.95)

train_tars = set(unique_tars[:train_cutoff])
val_tars = set(unique_tars[train_cutoff:val_cutoff])

def assign_split(tar_name):
    if tar_name in train_tars:
        return 'train'
    elif tar_name in val_tars:
        return 'val'
    return 'test'

df['split'] = df['tar_file'].apply(assign_split)
df.to_csv(PREPARED_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"✅ Prepared metadata saved to: {PREPARED_CSV_PATH}")

In [ ]:
# Cell 3: Hugging Face Login (Optional but avoids rate-limits)
from huggingface_hub import login
login()

In [ ]:
# Cell 4: 30% Stratified Staging & Manifest Engine
import os
import json
import tarfile
import warnings
import pandas as pd
import miniaudio
from tqdm import tqdm
from huggingface_hub import hf_hub_download

warnings.filterwarnings("ignore")

PROJECT_DIR = "/content/drive/MyDrive/farsi_asr_project"
HF_REPO_ID = "farsi-asr/ganjoor-chunked-asr-dataset"
METADATA_PATH = os.path.join(PROJECT_DIR, "ganjoor_prepared_metadata.csv")

STAGING_DIR = os.path.join(PROJECT_DIR, "staging_audio")
STAGING_MANIFEST_PATH = os.path.join(PROJECT_DIR, "staging_manifest.jsonl")
DONE_PATH = os.path.join(PROJECT_DIR, "completed_tars.txt")

os.makedirs(STAGING_DIR, exist_ok=True)

def get_audio_info(audio_bytes):
    try:
        info = miniaudio.get_file_info(audio_bytes)
        return info.duration, info.sample_rate
    except Exception:
        return 0.0, 0

def process_tar_worker(task):
    tar_filename, target_samples, archive_idx = task
    manifest_entries = []
    logs = {"success": 0, "failed": 0, "filtered": 0, "errors": []}

    try:
        tar_local_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=str(tar_filename).strip(),
            repo_type="dataset"
        )

        with tarfile.open(tar_local_path, "r:*") as tar:
            for member in tar.getmembers():
                filename = os.path.basename(member.name).strip()
                if filename in target_samples:
                    meta = target_samples[filename]
                    f = tar.extractfile(member)
                    if f is None:
                        logs["failed"] += 1
                        continue

                    audio_bytes = f.read()
                    duration, sample_rate = get_audio_info(audio_bytes)

                    if duration < 0.5 or duration > 30.0:
                        logs["filtered"] += 1
                        continue

                    safe_filename = f"{archive_idx}_{filename}"
                    staging_audio_path = os.path.join(STAGING_DIR, safe_filename)

                    with open(staging_audio_path, "wb") as out_f:
                        out_f.write(audio_bytes)

                    entry = {
                        "audio": staging_audio_path,
                        "text_clean": meta["text_clean"],
                        "duration": round(duration, 3),
                        "sample_rate": sample_rate,
                        "split": meta["split"],
                        "speaker_id": meta["speaker_id"],
                        "orig_tar": tar_filename
                    }
                    manifest_entries.append(entry)
                    logs["success"] += 1

        if os.path.exists(tar_local_path):
            os.remove(tar_local_path)

    except Exception as e:
        logs["errors"].append(f"Error on {tar_filename}: {str(e)}")

    return tar_filename, manifest_entries, logs

def run_phase1_staging(metadata_csv_path):
    print("Loading metadata...")
    df = pd.read_csv(metadata_csv_path)

    print("Applying 30% stratified sampling...")
    group_col = 'source_id' if 'source_id' in df.columns else 'split'
    df = df.groupby('split', group_keys=False).apply(
        lambda x: x.groupby(group_col, group_keys=False).sample(frac=0.3, random_state=42)
    )

    done_set = set()
    if os.path.exists(DONE_PATH):
        with open(DONE_PATH, "r", encoding="utf-8") as f:
            done_set = set(line.strip() for line in f if line.strip())
    print(f"Previously processed archives: {len(done_set)}")

    tasks = []
    for idx, (tar_file, sub_df) in enumerate(df.groupby('tar_file')):
        if str(tar_file).strip() in done_set:
            continue

        sub_map = {}
        for _, row in sub_df.iterrows():
            spk_id = str(row['source_id']).strip() if 'source_id' in row else "unknown"
            sub_map[str(row['audio']).strip()] = {
                "text_clean": str(row['text_clean']).strip(),
                "split": str(row['split']).strip(),
                "speaker_id": spk_id
            }
        tasks.append((tar_file, sub_map, idx))

    print(f"Remaining archives: {len(tasks)}")
    if not tasks:
        print("✅ All archives already processed!")
        return

    staging_manifest_writer = open(STAGING_MANIFEST_PATH, "a", encoding="utf-8")
    done_file_writer = open(DONE_PATH, "a", encoding="utf-8")

    for task in tqdm(tasks, desc="Staging Archives"):
        tar_filename, manifests, logs = process_tar_worker(task)

        if logs["success"] > 0:
            for entry in manifests:
                staging_manifest_writer.write(json.dumps(entry, ensure_ascii=False) + "\n")
            staging_manifest_writer.flush()
            done_file_writer.write(f"{tar_filename}\n")
            done_file_writer.flush()

    staging_manifest_writer.close()
    done_file_writer.close()
    print("✅ Staging phase completed successfully!")

run_phase1_staging(METADATA_PATH)